# 0. Pré-processamento do Dataset de Treinamento

Este notebook é a fonte única de preparação dos dados. Ele lê todos os CSVs em `files_brutos`, identifica o participante pelo texto antes do primeiro `_`, calcula ângulos por frame e gera datasets rastreáveis para os notebooks de treinamento.

Apenas arquivos existentes determinam os metadados. Não há tratamento especial para arquivos sintéticos nesta fase.

In [14]:
from datetime import datetime, timezone
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "files_brutos"
PROCESSED_DATA_DIR = PROJECT_ROOT / "files_processados"
UTILS_DIR = PROJECT_ROOT / "utils"

WINDOW_SIZE = 15
STRIDE = 1
TARGET_FPS = 5.0
MIN_VISIBILITY = 0.0
DATASET_VERSION = "v1"

ANGLE_COLUMNS = [
    "right_cotovelo",
    "left_cotovelo",
    "right_ombro",
    "left_ombro",
    "right_joelho",
    "left_joelho",
    "right_quadril",
    "left_quadril",
]
VISIBILITY_COLUMNS = [f"{column}_visibility_weight" for column in ANGLE_COLUMNS]

FRAMES_OUTPUT_PATH = PROCESSED_DATA_DIR / f"frames_processados_{DATASET_VERSION}.csv"
WINDOWS_OUTPUT_PATH = PROCESSED_DATA_DIR / f"janelas_w{WINDOW_SIZE}_s{STRIDE}_{DATASET_VERSION}.csv"
MANIFEST_OUTPUT_PATH = PROCESSED_DATA_DIR / f"manifesto_dataset_{DATASET_VERSION}.json"

assert RAW_DATA_DIR.is_dir(), f"Diretório não encontrado: {RAW_DATA_DIR}"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(UTILS_DIR))

print(f"Entrada: {RAW_DATA_DIR}")
print(f"Saída:   {PROCESSED_DATA_DIR}")
print(f"Janela:  {WINDOW_SIZE} frames | stride: {STRIDE} | FPS: {TARGET_FPS}")

Entrada: c:\Users\Rafael Luckner\Documents\repos\uniso\RACE - TCC\files_brutos
Saída:   c:\Users\Rafael Luckner\Documents\repos\uniso\RACE - TCC\files_processados
Janela:  15 frames | stride: 1 | FPS: 5.0


In [7]:
def build_file_metadata(csv_path: Path) -> dict:
    stem = csv_path.stem
    prefix, separator, _ = stem.partition("_")
    participant_id = prefix.strip().lower()
    validation_messages = []

    if not separator:
        validation_messages.append("nome sem underscore")
    if not participant_id:
        validation_messages.append("participant_id vazio")

    return {
        "video_id": stem,
        "participant_id": participant_id or pd.NA,
        "source_file": csv_path.name,
        "source_relative_path": csv_path.relative_to(PROJECT_ROOT).as_posix(),
        "file_stem": stem,
        "extension": csv_path.suffix.lower(),
        "participant_extraction_rule": "prefix_before_first_underscore",
        "file_validation_status": "valid" if not validation_messages else "warning",
        "file_validation_message": "; ".join(validation_messages) or pd.NA,
    }

csv_paths = sorted(
    path for path in RAW_DATA_DIR.glob("*.csv")
    if path.is_file() and not path.name.startswith(".")
)
file_metadata = pd.DataFrame([build_file_metadata(path) for path in csv_paths])

if file_metadata.empty:
    raise FileNotFoundError(f"Nenhum CSV encontrado em {RAW_DATA_DIR}")

file_metadata["participant_video_count"] = file_metadata.groupby("participant_id")["video_id"].transform("size")
file_metadata["generated_at_utc"] = datetime.now(timezone.utc).isoformat()

assert file_metadata["video_id"].is_unique, "video_id duplicado: renomeie os arquivos de origem."
print(f"CSVs elegíveis: {len(file_metadata)}")
display(file_metadata.groupby("participant_id", dropna=False)["video_id"].count().rename("videos").to_frame())

warnings = file_metadata.loc[file_metadata["file_validation_status"] != "valid"]
print(f"Avisos de nomenclatura: {len(warnings)}")
display(warnings)

CSVs elegíveis: 16


,videos
participant_id,
leticia,4
paulo,4
rafael,4
rodrigo,4


Avisos de nomenclatura: 0


,video_id,participant_id,source_file,source_relative_path,file_stem,extension,participant_extraction_rule,file_validation_status,file_validation_message,participant_video_count,generated_at_utc


In [8]:
def load_raw_landmarks(file_row: pd.Series) -> pd.DataFrame:
    csv_path = PROJECT_ROOT / file_row["source_relative_path"]
    raw = pd.read_csv(csv_path)
    required_columns = {
        "frame", "timestamp_s", "landmark_idx", "x", "y", "visibility", "exercise"
    }
    missing_columns = required_columns - set(raw.columns)
    if missing_columns:
        raise ValueError(f"{csv_path.name}: colunas ausentes: {sorted(missing_columns)}")

    exercise_values = raw["exercise"].dropna().unique()
    if len(exercise_values) != 1:
        raise ValueError(f"{csv_path.name}: esperado um exercício, encontrados {exercise_values.tolist()}")

    raw = raw.copy()
    for field in [
        "video_id", "participant_id", "source_file", "source_relative_path",
        "participant_extraction_rule", "file_validation_status"
    ]:
        raw[field] = file_row[field]
    raw["exercise_label"] = exercise_values[0]
    return raw

raw_landmarks = pd.concat(
    [load_raw_landmarks(row) for _, row in file_metadata.iterrows()],
    ignore_index=True,
)

print(f"Landmarks carregados: {len(raw_landmarks):,}")
print(f"Vídeos: {raw_landmarks['video_id'].nunique()} | Participantes: {raw_landmarks['participant_id'].nunique()}")
display(
    raw_landmarks.groupby(["participant_id", "exercise_label"], dropna=False)["frame"]
    .nunique()
    .rename("frames")
    .reset_index()
    .sort_values(["participant_id", "exercise_label"])
)

Landmarks carregados: 496,402
Vídeos: 16 | Participantes: 4


,participant_id,exercise_label,frames
0,leticia,agachamento,784
1,leticia,descanso,866
2,leticia,flexao,589
3,leticia,rosca_biceps,630
4,paulo,agachamento,499
5,paulo,descanso,608
6,paulo,flexao,509
7,paulo,rosca_biceps,691
8,rafael,agachamento,1587
9,rafael,descanso,1740


In [15]:
from utils.calculador_angulos import extract_angles_from_landmarks

# A função compartilhada usa `video_source` para separar frames de vídeos distintos.
landmarks_for_angles = raw_landmarks.copy()
landmarks_for_angles["video_source"] = landmarks_for_angles["video_id"]
angles_by_frame = extract_angles_from_landmarks(landmarks_for_angles).rename(
    columns={"video_source": "video_id"}
)

frame_metadata_columns = [
    "video_id", "participant_id", "source_file", "source_relative_path",
    "exercise_label", "participant_extraction_rule", "file_validation_status",
]
frame_metadata = raw_landmarks[frame_metadata_columns + ["frame"]].drop_duplicates(
    ["video_id", "frame"]
)

frames_processed = angles_by_frame.merge(
    frame_metadata,
    on=["video_id", "frame"],
    how="inner",
    validate="one_to_one",
).sort_values(["participant_id", "video_id", "frame"]).reset_index(drop=True)

frames_processed = frames_processed.drop(columns=["exercise"], errors="ignore")
frames_processed["valid_angle_count"] = frames_processed[ANGLE_COLUMNS].notna().sum(axis=1)
frames_processed["mean_visibility"] = frames_processed[VISIBILITY_COLUMNS].mean(axis=1)
frames_processed["is_valid_frame"] = (
    frames_processed["valid_angle_count"].eq(len(ANGLE_COLUMNS))
    & frames_processed["mean_visibility"].ge(MIN_VISIBILITY)
)
frames_processed["dataset_version"] = DATASET_VERSION
frames_processed["generated_at_utc"] = datetime.now(timezone.utc).isoformat()

print(f"Frames processados: {len(frames_processed):,}")
print(f"Frames válidos: {frames_processed['is_valid_frame'].mean():.2%}")
display(frames_processed.head())

Frames processados: 15,042
Frames válidos: 100.00%


,frame,timestamp_s,video_id,right_cotovelo,right_cotovelo_visibility_weight,left_cotovelo,left_cotovelo_visibility_weight,right_ombro,right_ombro_visibility_weight,left_ombro,...,source_file,source_relative_path,exercise_label,participant_extraction_rule,file_validation_status,valid_angle_count,mean_visibility,is_valid_frame,dataset_version,generated_at_utc
0,0,0.0,leticia_Descanso720p (3)_full_5fps_descanso,179.881302,0.987399,179.457504,0.995632,14.215514,0.995659,16.781509,...,leticia_Descanso720p (3)_full_5fps_descanso.csv,files_brutos/leticia_Descanso720p (3)_full_5fp...,descanso,prefix_before_first_underscore,valid,8,0.995065,True,v1,2026-09-21T23:38:35.846666+00:00
1,6,0.2,leticia_Descanso720p (3)_full_5fps_descanso,169.640884,0.987146,171.894516,0.995222,13.882917,0.995592,15.822812,...,leticia_Descanso720p (3)_full_5fps_descanso.csv,files_brutos/leticia_Descanso720p (3)_full_5fp...,descanso,prefix_before_first_underscore,valid,8,0.994856,True,v1,2026-09-21T23:38:35.846666+00:00
2,12,0.4,leticia_Descanso720p (3)_full_5fps_descanso,169.854980,0.986294,151.637955,0.995272,14.664306,0.994949,14.834850,...,leticia_Descanso720p (3)_full_5fps_descanso.csv,files_brutos/leticia_Descanso720p (3)_full_5fp...,descanso,prefix_before_first_underscore,valid,8,0.994385,True,v1,2026-09-21T23:38:35.846666+00:00
3,18,0.6,leticia_Descanso720p (3)_full_5fps_descanso,178.208221,0.984511,164.142563,0.994849,14.034078,0.994393,12.852586,...,leticia_Descanso720p (3)_full_5fps_descanso.csv,files_brutos/leticia_Descanso720p (3)_full_5fp...,descanso,prefix_before_first_underscore,valid,8,0.993722,True,v1,2026-09-21T23:38:35.846666+00:00
4,24,0.8,leticia_Descanso720p (3)_full_5fps_descanso,178.606598,0.984505,175.621216,0.994941,17.319216,0.994368,13.706882,...,leticia_Descanso720p (3)_full_5fps_descanso.csv,files_brutos/leticia_Descanso720p (3)_full_5fp...,descanso,prefix_before_first_underscore,valid,8,0.993750,True,v1,2026-09-21T23:38:35.846666+00:00


In [12]:
def create_window_dataset(
    frames: pd.DataFrame,
    window_size: int,
    stride: int,
    angle_columns: list[str],
) -> pd.DataFrame:
    metadata_columns = [
        "video_id",
        "participant_id",
        "source_file",
        "source_relative_path",
        "exercise_label",
        "participant_extraction_rule",
        "file_validation_status",
    ]
    window_rows = []

    for video_id, group in frames.groupby("video_id", sort=False):
        group = group.sort_values("frame").reset_index(drop=True)
        valid_group = group.loc[group["is_valid_frame"]].reset_index(drop=True)

        for start_index in range(0, len(valid_group) - window_size + 1, stride):
            window = valid_group.iloc[start_index : start_index + window_size]
            first_frame = window.iloc[0]
            row = {
                "window_id": f"{video_id}__start_{int(first_frame['frame'])}__w{window_size}__s{stride}",
                "window_size": window_size,
                "stride": stride,
                "window_start_frame": int(first_frame["frame"]),
                "window_end_frame": int(window.iloc[-1]["frame"]),
                "window_start_timestamp_s": float(first_frame["timestamp_s"]),
                "window_end_timestamp_s": float(window.iloc[-1]["timestamp_s"]),
                "window_duration_s": float(window.iloc[-1]["timestamp_s"] - first_frame["timestamp_s"]),
                "valid_frame_ratio": float(window["is_valid_frame"].mean()),
                "mean_visibility": float(window["mean_visibility"].mean()),
                "min_visibility": float(window["mean_visibility"].min()),
                "dataset_version": DATASET_VERSION,
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            row.update(first_frame[metadata_columns].to_dict())

            for frame_position, (_, frame_data) in enumerate(window.iterrows(), start=1):
                for angle_column in angle_columns:
                    row[f"frame_{frame_position}_{angle_column}"] = frame_data[angle_column]

            window_rows.append(row)

    windows = pd.DataFrame(window_rows)
    if windows.empty:
        raise ValueError("Nenhuma janela foi criada. Revise WINDOW_SIZE, STRIDE e a qualidade dos frames.")
    if not windows["window_id"].is_unique:
        raise ValueError("window_id duplicado. Revise os identificadores de vídeo e de frame.")
    return windows

windows_processed = create_window_dataset(
    frames_processed,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    angle_columns=ANGLE_COLUMNS,
)

feature_columns = [
    f"frame_{frame_position}_{angle_column}"
    for frame_position in range(1, WINDOW_SIZE + 1)
    for angle_column in ANGLE_COLUMNS
]

assert windows_processed[feature_columns].notna().all().all(), "Há features ausentes nas janelas."
print(f"Janelas processadas: {len(windows_processed):,}")
print(f"Features por janela: {len(feature_columns)}")
display(windows_processed.head())

Janelas processadas: 14,818
Features por janela: 120


,window_id,window_size,stride,window_start_frame,window_end_frame,window_start_timestamp_s,window_end_timestamp_s,window_duration_s,valid_frame_ratio,mean_visibility,...,frame_14_right_quadril,frame_14_left_quadril,frame_15_right_cotovelo,frame_15_left_cotovelo,frame_15_right_ombro,frame_15_left_ombro,frame_15_right_joelho,frame_15_left_joelho,frame_15_right_quadril,frame_15_left_quadril
0,leticia_Descanso720p (3)_full_5fps_descanso__s...,15,1,0,84,0.0,2.8,2.8,1.0,0.984836,...,173.778824,178.840714,115.168320,125.998375,11.368341,9.700243,177.669296,176.548538,174.683884,179.886353
1,leticia_Descanso720p (3)_full_5fps_descanso__s...,15,1,6,90,0.2,3.0,2.8,1.0,0.980613,...,174.683884,179.886353,126.026711,134.915619,9.900197,10.689015,176.678146,177.161011,174.287918,178.905655
2,leticia_Descanso720p (3)_full_5fps_descanso__s...,15,1,12,96,0.4,3.2,2.8,1.0,0.975627,...,174.287918,178.905655,137.900558,136.281921,6.285844,10.925443,170.649628,177.279327,171.188660,178.333389
3,leticia_Descanso720p (3)_full_5fps_descanso__s...,15,1,18,102,0.6,3.4,2.8,1.0,0.969997,...,171.188660,178.333389,132.603821,133.028580,6.925603,11.506891,177.480438,177.607788,170.989655,178.160904
4,leticia_Descanso720p (3)_full_5fps_descanso__s...,15,1,24,108,0.8,3.6,2.8,1.0,0.963878,...,170.989655,178.160904,127.784775,145.822342,9.046068,5.861502,178.192230,177.053223,170.726761,176.852890


In [ ]:
import json

frames_processed.to_csv(FRAMES_OUTPUT_PATH, index=False)
windows_processed.to_csv(WINDOWS_OUTPUT_PATH, index=False)

manifest = {
    "dataset_version": DATASET_VERSION,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_directory": RAW_DATA_DIR.relative_to(PROJECT_ROOT).as_posix(),
    "window_configuration": {
        "window_size": WINDOW_SIZE,
        "stride": STRIDE,
        "target_fps": TARGET_FPS,
        "min_visibility": MIN_VISIBILITY,
    },
    "files": {
        "frames": FRAMES_OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix(),
        "windows": WINDOWS_OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix(),
    },
    "counts": {
        "participants": int(file_metadata["participant_id"].nunique()),
        "videos": int(file_metadata["video_id"].nunique()),
        "frames": int(len(frames_processed)),
        "windows": int(len(windows_processed)),
        "features_per_window": len(feature_columns),
    },
    "required_window_columns": [
        "window_id", "video_id", "participant_id", "exercise_label",
        "window_size", "stride", "window_start_frame", "window_end_frame",
        "mean_visibility", "valid_frame_ratio", *feature_columns,
    ],
}
MANIFEST_OUTPUT_PATH.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Datasets salvos:")
for output_path in [FRAMES_OUTPUT_PATH, WINDOWS_OUTPUT_PATH, MANIFEST_OUTPUT_PATH]:
    print(f"  {output_path.relative_to(PROJECT_ROOT)} ({output_path.stat().st_size / 1024:.1f} KB)")

Datasets salvos:
  files_processados\frames_processados_v1.csv (7842.1 KB)
  files_processados\janelas_w15_s1_v1.csv (38179.9 KB)
  files_processados\manifesto_dataset_v1.json (4.2 KB)
